## Product recommendation system
build a simple recommender that suggests products often bought together using grocery store transaction data.

In [107]:
import pandas as pd

In [108]:
df = pd.read_csv("../data/processed/transactions_sample.csv")

In [109]:
df.head()

,order_id,product_id,add_to_cart_order,reordered,product_name
0,3109255,34099,16,0,Crushed Red Chili Pepper
1,301098,41950,5,0,Organic Tomato Cluster
2,1181866,45066,8,0,Honeycrisp Apple
3,1678630,8859,2,1,Natural Spring Water
4,644090,24781,2,0,"PODS Laundry Detergent, Ocean Mist Designed fo..."


In [110]:
df.shape

(200000, 5)

Identify the most frequently purchased products.

In [111]:
product_counts = df["product_name"].value_counts()
product_counts.head(10)

product_name
Banana                    2976
Bag of Organic Bananas    2333
Organic Strawberries      1668
Organic Baby Spinach      1489
Organic Hass Avocado      1301
Organic Avocado           1081
Large Lemon                917
Organic Raspberries        901
Strawberries               862
Organic Whole Milk         853
Name: count, dtype: int64

## Limit number of products
Keep only the most frequently purchased products to reduce computation.

In [112]:
top_products = product_counts.head(500).index

Filter the dataset to include only the selected popular products.

In [113]:
df_filtered = df[df["product_name"].isin(top_products)]

In [114]:
df_filtered.shape

(85976, 5)

## Basket matrix
Create a matrix where:
- rows represent orders
- columns represent products
- values indicate whether the product appears in the order

In [115]:
basket_matrix = pd.crosstab(
    df_filtered["order_id"],
    df_filtered["product_name"]
)

In [116]:
basket_matrix.head()

product_name,0% Greek Strained Yogurt,1% Lowfat Milk,100 Calorie Per Bag Popcorn,100% Raw Coconut Water,100% Recycled Paper Towels,100% Whole Wheat Bread,2% Reduced Fat DHA Omega-3 Reduced Fat Milk,2% Reduced Fat Milk,2% Reduced Fat Organic Milk,85% Lean Ground Beef,...,Whole Milk Plain Yogurt,Whole Milk Ricotta Cheese,XL Emerald White Seedless Grapes,Yellow Bell Pepper,Yellow Onions,YoKids Blueberry & Strawberry/Vanilla Yogurt,"YoKids Squeezers Organic Low-Fat Yogurt, Strawberry",Yobaby Organic Plain Yogurt,"Yogurt, Strained Low-Fat, Coconut",Zero Calorie Cola
order_id,,,,,,,,,,,,,,,,,,,,,
61,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
71,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
129,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
147,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
241,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Count how often products appear together in the same order.

In [117]:
co_occurrence_matrix = basket_matrix.T.dot(basket_matrix)

In [118]:
co_occurrence_matrix.shape

(500, 500)

In [119]:
for i in range(len(co_occurrence_matrix)):
    co_occurrence_matrix.iat[i, i] = 0

In [120]:
co_occurrence_matrix.head()

product_name,0% Greek Strained Yogurt,1% Lowfat Milk,100 Calorie Per Bag Popcorn,100% Raw Coconut Water,100% Recycled Paper Towels,100% Whole Wheat Bread,2% Reduced Fat DHA Omega-3 Reduced Fat Milk,2% Reduced Fat Milk,2% Reduced Fat Organic Milk,85% Lean Ground Beef,...,Whole Milk Plain Yogurt,Whole Milk Ricotta Cheese,XL Emerald White Seedless Grapes,Yellow Bell Pepper,Yellow Onions,YoKids Blueberry & Strawberry/Vanilla Yogurt,"YoKids Squeezers Organic Low-Fat Yogurt, Strawberry",Yobaby Organic Plain Yogurt,"Yogurt, Strained Low-Fat, Coconut",Zero Calorie Cola
product_name,,,,,,,,,,,,,,,,,,,,,
0% Greek Strained Yogurt,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1% Lowfat Milk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
100 Calorie Per Bag Popcorn,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
100% Raw Coconut Water,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
100% Recycled Paper Towels,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Recommendation function


In [121]:
def recommend_products(product_name, n=5):
    if product_name not in co_occurrence_matrix.columns:
        return f"Product '{product_name}' not found."
    
    similar_products = co_occurrence_matrix[product_name].sort_values(
        ascending=False
    )
    return similar_products.head(n)

In [122]:
recommend_products("Banana")

product_name
Cucumber Kirby                       5
Organic Garnet Sweet Potato (Yam)    3
Honeycrisp Apple                     3
Large Lemon                          3
Apple Honeycrisp Organic             3
Name: Banana, dtype: int64

In [123]:
recommend_products("Organic Strawberries")

product_name
Spring Water              3
Banana                    2
Bag of Organic Bananas    2
Original Hummus           2
Organic Salted Butter     2
Name: Organic Strawberries, dtype: int64

## Summary
Pipeline implemented in this notebook:
- select popular products
- build basket matrix
- compute product co-occurrence
- generate product recommendations